# E1 · pd.get_dummies on training and test: same size, different columns

**Type**: Find the bug · **Level**: ●●○ independent · **Time**: about 30 minutes · **core path ★**

## Objective

Find out why a model with an excellent AUC during development produces predictions worse than chance on new data, even though there is
no error or warning: the one-hot columns of the test set do not match those of the training set. Fix it in two ways.

## Dataset

Synthetic, from `generate(n, regions, seed)`: a categorical `region` and a numeric feature `x`, with different true effects for
each region (`Center +2`, `North −2`, `South 0`, `Sardinia +1`). The **training set** contains Center, North and South. The **test set** (which
simulates the production data) contains North, South and **Sardinia**: a new region, and a known one (Center) that does not show up.

## Educational purpose

Recognize one of the most common silent bugs: `pd.get_dummies` decides the columns by looking at **the data it receives**, not at those of the
training set. With the same number of columns and a different order the model applies every coefficient to the wrong column.

## Task

1. Run the colleague's cell: look at the shapes of `Xd_tr` and `Xd_te`. They are equal: no error. Save in `different_columns` `True` if the lists of column names of the two DataFrames are different.
2. Fix it with a `OneHotEncoder(handle_unknown="ignore", sparse=False)` fitted on the training set alone, on the `region` column, concatenated with `x`. Save `auc_ok`.
3. Also fix it in a second way, staying in pandas: `pd.get_dummies` on the test set and then `reindex(columns=Xd_tr.columns, fill_value=0)`. Save `auc_reindex`.
4. Explain in `answer` ("a", "b" or "c") why the bug raises no errors:
   - **a**: the number of columns matches (4), so the input has the right shape, but the names (and the order) of the columns do not, and the model (in scikit-learn 1.1) uses only the positions;
   - **b**: `LogisticRegression` ignores categorical variables;
   - **c**: `pd.get_dummies` fixes the columns on first use.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OneHotEncoder


def generate(n, regions, seed):
    r = np.random.default_rng(seed)
    reg = r.choice(regions, n)
    eff = {"Center": 2.0, "North": -2.0, "South": 0.0, "Sardinia": 1.0}
    x = r.normal(size=n)
    logit = np.array([eff[k] for k in reg]) + 1.0 * x
    return pd.DataFrame({"region": reg, "x": x}), r.binomial(1, 1 / (1 + np.exp(-logit)))


X_tr, y_tr = generate(800, ["Center", "North", "South"], 1)
X_te, y_te = generate(300, ["North", "Sardinia", "South"], 2)      # in production: a new region, and a known one that does not show up

## Your work

### The colleague's code

In [ ]:
Xd_tr = pd.get_dummies(X_tr)
Xd_te = pd.get_dummies(X_te)
print(Xd_tr.shape, Xd_te.shape)                       # same size: no error

model = LogisticRegression(max_iter=1000).fit(Xd_tr.to_numpy(dtype=float), y_tr)
auc_bug = roc_auc_score(y_te, model.predict_proba(Xd_te.to_numpy(dtype=float))[:, 1])
print(f"test AUC: {auc_bug:.3f}")

### Your diagnosis and the fixes

In [ ]:
different_columns = None
auc_ok = auc_bug
auc_reindex = auc_bug
answer = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`ohe = OneHotEncoder(handle_unknown="ignore", sparse=False).fit(X_tr[["region"]])` and then `np.column_stack([X[["x"]].to_numpy(), ohe.transform(X[["region"]])])`. For the reindex the column order must be that of `Xd_tr.columns`; the columns that do not show up in the test set are filled with zeros and the new ones (`region_Sardinia`) discarded.'):
    check(different_columns is True and list(Xd_tr.columns) != list(Xd_te.columns), "the columns of the test set are not those of the training set",
          "compare list(Xd_tr.columns) with list(Xd_te.columns): different_columns must be True")
    check(auc_ok > auc_bug + 0.05, "the OneHotEncoder fitted on the training set clearly improves the test AUC",
          "auc_ok must come from an encoder fitted ONLY on X_tr and applied with transform to X_te")
    check(abs(auc_ok - auc_reindex) < 1e-9, "the reindex on the training columns is equivalent to the OneHotEncoder",
          "get_dummies(X_te).reindex(columns=Xd_tr.columns, fill_value=0) must give the same result")
    check(answer == "a", "correct explanation", "reread: what changes between the two matrices, besides the number of columns?")
done()

---
The commented solution is in `solutions/E_find_the_bug/E1_get_dummies_misaligned_columns.ipynb`. Open it after you pass the check, or after you have really tried.